In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# Récupère la session Spark active
spark = SparkSession.builder.getOrCreate()

# Récupère dbutils de façon compatible (Databricks SDK / PySpark)
try:
    from databricks.sdk.runtime import dbutils
except ImportError:
    try:
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)
    except ImportError:
        pass  # En environnement Databricks natif, dbutils est déjà injecté

spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA silver")

# Lecture streaming depuis Bronze
bronze_stream = (
    spark.readStream
    .table("main.bronze.transactions_bronze_stream")
)

silver_stream = (
    bronze_stream
    # Nettoyage des valeurs nulles
    .na.drop(subset=["Amount", "Class"])

    # Typage
    .withColumn("Amount", F.col("amount").cast("double"))
    .withColumn("is_fraud", F.col("Class").cast("int"))

    # Suppression des doublons
    .dropDuplicates()

    # Suppression de la colonne Class
    .drop("Class")

    # Renommer les colonnes
    .withColumnRenamed("Amount", "amount")
    .withColumnRenamed("Time", "time")
)

# Ecriture streaming vers la table Silver
silver_checkpoint = (
    "/Volumes/main/silver/silver_volume/"
    "_checkpoints/silver_stream"
)

query = (
    silver_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", silver_checkpoint)
    .trigger(once=True)
    .table("transactions_silver_stream")
)